# PHASE 1: DATA LOADING & VOLUME-WISE SPLIT

**Objective:** Scan 2D PNG data, build volume index, create train/val/test splits (volume-wise to prevent data leakage)

**Author:** [Your Name]
**Date:** 2026-05-20
**Hardware:** NVIDIA RTX 3050 Ti 4GB

---

## Table of Contents
1. GPU Setup & Import
2. Data Path Configuration
3. Build Volume Index
4. Dataset Statistics
5. Volume-Wise Split
6. Save Split Files
7. Verify No Data Leakage
8. Final Summary

## [SETUP] Cell 1: GPU Setup & Import src/ modules

In [ ]:
import os
import sys
import json
import random
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Ensure src/ is importable (walk up from cwd until src/ found)
project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import from src/
from src.data_loader import (
    DatasetConfig,
    DataPathManager,
    VolumeWiseSplitter,
    create_2d_dataloaders,
)
from src.gpu_utils import setup_device, gpu_report, print_gpu_memory

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# GPU Configuration
DEVICE = setup_device()
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA: {torch.version.cuda}")
    torch.backends.cudnn.benchmark = True
else:
    print("[WARNING] Using CPU (slower)")
print("=" * 60)

## [CONFIG] Cell 2: Data Path Configuration

In [ ]:
# Configure paths using DatasetConfig
print("[INFO] Dataset Configuration:")
print(f"   Images Directory: {DatasetConfig.IMAGES_DIR}")
print(f"   Masks Directory: {DatasetConfig.MASKS_DIR}")
print(f"   Output Directory: {DatasetConfig.OUTPUT_DIR}")

# Verify directories exist
img_dir = DatasetConfig.IMAGES_DIR
mask_dir = DatasetConfig.MASKS_DIR

assert img_dir.exists(), f"Images directory not found: {img_dir}"
assert mask_dir.exists(), f"Masks directory not found: {mask_dir}"

print("[OK] Directories verified")

## [INDEX] Cell 3: Build Volume Index

In [ ]:
# Build volume index by scanning directories
path_manager = DataPathManager()
volume_index = path_manager.build_index()

print(f"[INFO] Volume Index Built:")
print(f"   Total volumes: {len(volume_index['volumes'])}")
print(f"   Volume IDs: {volume_index['volumes'][:5]}... (showing first 5)")

# Show sample slices per volume
print("\n[INFO] Sample slice counts:")
for vol_id in sorted(volume_index['image_paths'].keys())[:5]:
    num_slices = len(volume_index['image_paths'][vol_id])
    has_mask = vol_id in volume_index['mask_paths']
    print(f"   Volume {vol_id}: {num_slices} slices, has_mask={has_mask}")

## [STATS] Cell 4: Dataset Statistics

In [ ]:
# Compute dataset statistics
all_image_ids = set(volume_index['image_paths'].keys())
all_mask_ids = set(volume_index['mask_paths'].keys())
total_volumes = len(all_image_ids)
total_slices = sum(len(v) for v in volume_index['image_paths'].values())
volumes_with_masks = len(all_image_ids & all_mask_ids)
volumes_without_masks = len(all_image_ids - all_mask_ids)

print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
print(f"Total Volumes:        {total_volumes}")
print(f"Total Slices:          {total_slices:,}")
print(f"Volumes with Masks:    {volumes_with_masks}")
print(f"Volumes without Masks:{volumes_without_masks}")
print("=" * 60)

# Compute volume-wise statistics inline
volume_rows = []
for vid in sorted(volume_index['image_paths'].keys()):
    num_slices = len(volume_index['image_paths'][vid])
    has_mask = vid in volume_index['mask_paths']
    volume_rows.append({'volume_id': vid, 'num_slices': num_slices, 'has_mask': has_mask})
volume_stats = {'volumes': volume_rows, 'total_slices': total_slices}

# Create summary dataframe
df_volumes = pd.DataFrame(volume_stats['volumes'])
print("\n[INFO] Volume Statistics:")
print(f"   Min slices per volume: {df_volumes['num_slices'].min()}")
print(f"   Max slices per volume: {df_volumes['num_slices'].max()}")
print(f"   Mean slices per volume: {df_volumes['num_slices'].mean():.1f}")

## [SPLIT] Cell 5: Volume-Wise Split

In [ ]:
# Create volume-wise splitter (80/10/10 split)
splitter = VolumeWiseSplitter(split_ratios=(0.8, 0.1, 0.1))

# Get split
splits = splitter.split(volume_index['volumes'])

print("=" * 60)
print("VOLUME SPLIT CONFIGURATION")
print("=" * 60)
print(f"Train: {len(splits['train'])} volumes ({splits['train'][0]}-{splits['train'][-1]})")
print(f"Val:    {len(splits['val'])} volumes ({splits['val'][0]}-{splits['val'][-1]})")
print(f"Test:   {len(splits['test'])} volumes ({splits['test'][0]}-{splits['test'][-1]})")
print("=" * 60)

# Calculate slice counts per split
train_slices = sum(len(volume_index['image_paths'].get(v, [])) for v in splits['train'])
val_slices = sum(len(volume_index['image_paths'].get(v, [])) for v in splits['val'])
test_slices = sum(len(volume_index['image_paths'].get(v, [])) for v in splits['test'])

print("\n[INFO] Slice distribution:")
print(f"   Train: {train_slices:,} slices ({train_slices/volume_stats['total_slices']*100:.1f}%)")
print(f"   Val:   {val_slices:,} slices ({val_slices/volume_stats['total_slices']*100:.1f}%)")
print(f"   Test:  {test_slices:,} slices ({test_slices/volume_stats['total_slices']*100:.1f}%)")

## [SAVE] Cell 6: Save Split Files

In [ ]:
# Ensure output directory exists
splits_dir = DatasetConfig.SPLITS_DIR
splits_dir.mkdir(parents=True, exist_ok=True)

# Save split files (comma-separated volume IDs)
for split_name, vol_ids in splits.items():
    filepath = splits_dir / f"{split_name}_volumes.txt"
    with open(filepath, 'w') as f:
        f.write(','.join(map(str, vol_ids)))
    print(f"[OK] Saved {split_name} split: {filepath}")

# Verify files were created
print("\n[INFO] Split files created:")
for split_name in ['train', 'val', 'test']:
    filepath = splits_dir / f"{split_name}_volumes.txt"
    if filepath.exists():
        with open(filepath, 'r') as f:
            content = f.read().strip()
        print(f"   {split_name}: {len(content.split(','))} volumes")

## [VERIFY] Cell 7: Verify No Data Leakage

In [ ]:
# Verify no overlap between splits (data leakage check)
train_set = set(splits['train'])
val_set = set(splits['val'])
test_set = set(splits['test'])

train_val_overlap = train_set & val_set
train_test_overlap = train_set & test_set
val_test_overlap = val_set & test_set

print("=" * 60)
print("DATA LEAKAGE CHECK")
print("=" * 60)
if not train_val_overlap and not train_test_overlap and not val_test_overlap:
    print("[OK] No data leakage detected!")
    print(f"   Train-Val overlap: {len(train_val_overlap)}")
    print(f"   Train-Test overlap: {len(train_test_overlap)}")
    print(f"   Val-Test overlap: {len(val_test_overlap)}")
else:
    print("[ERROR] Data leakage detected!")
    print(f"   Train-Val overlap: {train_val_overlap}")
    print(f"   Train-Test overlap: {train_test_overlap}")
    print(f"   Val-Test overlap: {val_test_overlap}")
print("=" * 60)

## [TEST] Cell 8: Test DataLoader Creation

In [ ]:
# Test creating dataloaders (small sample for verification)
train_vids = splits['train'][:5]  # Just 5 volumes for test
val_vids = splits['val'][:2]

train_loader, val_loader, _ = create_2d_dataloaders(
    volume_index,
    train_vids,
    val_vids,
    [],
    batch_size=4,
    num_workers=0,
    pin_memory=False,
)

print("[NEW] DataLoader test:")
for batch in train_loader:
    imgs = batch['image']
    masks = batch['mask']
    print(f"   Batch: imgs={imgs.shape}, masks={masks.shape}")
    print(f"   Img range: [{imgs.min():.3f}, {imgs.max():.3f}]")
    print(f"   Mask range: [{masks.min():.3f}, {masks.max():.3f}]")
    break

print("[OK] DataLoaders working correctly")

## [VISUAL] Cell 9: Visualize Split Distribution

In [ ]:
# Visualize volume distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Slice counts per volume
volumes = sorted(volume_index['image_paths'].keys())
slice_counts = [len(volume_index['image_paths'][v]) for v in volumes]

colors = ['green' if v in splits['train'] else 'orange' if v in splits['val'] else 'red' for v in volumes]
axes[0].bar(range(len(volumes)), slice_counts, color=colors)
axes[0].set_xlabel('Volume Index')
axes[0].set_ylabel('Number of Slices')
axes[0].set_title('Slice Counts per Volume (Train=Green, Val=Orange, Test=Red)')

# Plot 2: Split pie chart
split_sizes = [train_slices, val_slices, test_slices]
labels = [f'Train\n{train_slices:,} ({train_slices/volume_stats["total_slices"]*100:.1f}%)',
           f'Val\n{val_slices:,} ({val_slices/volume_stats["total_slices"]*100:.1f}%)',
           f'Test\n{test_slices:,} ({test_slices/volume_stats["total_slices"]*100:.1f}%)']
axes[1].pie(split_sizes, labels=labels, colors=['green', 'orange', 'red'], startangle=90)
axes[1].set_title('Dataset Split Distribution')

plt.tight_layout()
output_fig_dir = DatasetConfig.PROJECT_DIR / "outputs" / "data_loading"
output_fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = output_fig_dir / 'split_visualization.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"[OK] Visualization saved to {fig_path}")

## [SUMMARY] Cell 10: Final Summary

In [ ]:
# Save summary metadata
metadata = {
    'total_volumes': total_volumes,
    'total_slices': total_slices,
    'volumes_with_masks': volumes_with_masks,
    'volumes_without_masks': volumes_without_masks,
    'train_volumes': splits['train'],
    'val_volumes': splits['val'],
    'test_volumes': splits['test'],
    'train_slices': train_slices,
    'val_slices': val_slices,
    'test_slices': test_slices,
}

# Save metadata
DatasetConfig.METADATA_DIR.mkdir(parents=True, exist_ok=True)
metadata_path = DatasetConfig.METADATA_DIR / 'phase1_summary.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("=" * 60)
print("PHASE 1 COMPLETE")
print("=" * 60)
print(f"Dataset: {total_volumes} volumes, {total_slices:,} slices")
print(f"Split:   Train={len(splits['train'])}v, Val={len(splits['val'])}v, Test={len(splits['test'])}v")
print(f"Data leakage: NONE")
print(f"Split files: {splits_dir}")
print(f"Metadata: {metadata_path}")
print("=" * 60)